# 🧱 Part 14: RLHF Alignment: Turning Preferences Into Objectives

> **Previous context**: Training predicts tokens, but useful assistants must optimize for helpful behavior.
> **Goal for this part**: Understand Reward Models, Bradley-Terry preference loss, PPO clipping, KL penalty, and DPO intuition.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. The mismatch

Pretraining teaches continuation. Users want helpful, honest, safe answers. Alignment methods turn preference data into training signals.

## 1. Reward model

A reward model learns to score which response humans prefer. Pairwise comparisons become a supervised learning problem.

## 2. PPO and KL

PPO updates the policy toward higher reward while clipping changes. KL penalty keeps the new model close to the reference model.

## 3. DPO

DPO removes the explicit reward-model step and directly optimizes chosen responses over rejected responses.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] RLHF uses preference data.
- [ ] Reward models score responses.
- [ ] PPO and DPO are two ways to move a model toward preferred behavior.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [1]:
import numpy as np
import math

np.random.seed(42)

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.       │
       ▼  Stage 1: SFT
  ┌─────────────────────────┐
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  └────────────┬────────────┘
               ▼
Read the values printed above and connect them to the concept in this cell.               │
         ┌─────┴─────┐
         ▼             ▼
    Stage 2a: RLHF    Stage 2b: DPO
Read the values printed above and connect them to the concept in this cell.         │                  │
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.         │                  │
         └──────┬───────────┘
                ▼
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

---

### Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Loss
Read the values printed above and connect them to the concept in this cell.1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

#### Loss
Read the values printed above and connect them to the concept in this cell.
```
[system prompt] [user message] [assistant response]
```

Read the values printed above and connect them to the concept in this cell.
Loss
Read the values printed above and connect them to the concept in this cell.- Loss- Read the values printed above and connect them to the concept in this cell.
Loss

In [ ]:
# Teaching note: follow this line to see the main step.
print('Loss')
print()

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.

tokens = [1, 5, 3, 8, 2, 7, 9, 4, 6]  # Teaching note: follow this line to see the main step.
prompt_len = 5  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
labels_shifted = tokens[1:] + [-1]  # [5, 3, 8, 2, 7, 9, 4, 6, -1]

# Teaching note: follow this line to see the main step.
labels_masked = labels_shifted.copy()
for i in range(prompt_len):
    labels_masked[i] = -100  # Teaching note: follow this line to see the main step.

print("tokens:      ", tokens)
print('Read the values printed above and connect them to the concept in this cell.', labels_shifted)
print("labels(mask):", labels_masked)
print()

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
np.random.seed(42)
log_probs = np.random.uniform(-3.0, -0.5, size=len(tokens))
log_probs = np.round(log_probs, 2)

print('Loss')
total_loss = 0
count = 0
for i in range(len(tokens) - 1):  # Teaching note: follow this line to see the main step.
    label = labels_shifted[i]
    loss_i = -log_probs[i]  # Teaching note: follow this line to see the main step.
    total_loss += loss_i
    count += 1
    print(f"Read the values printed above and connect them to the concept in this cell.{i}: token={tokens[i]}, label={label:>3}, "
          f"log_p={log_probs[i]:.2f}, loss={loss_i:.2f}")
avg_no_mask = total_loss / count
print(f"Loss{avg_no_mask:.4f}Read the values printed above and connect them to the concept in this cell.{count}Read the values printed above and connect them to the concept in this cell.")
print()

print('Loss')
total_loss = 0
count = 0
for i in range(len(tokens) - 1):
    label = labels_masked[i]
    if label == -100:
        print(f"Read the values printed above and connect them to the concept in this cell.{i}: token={tokens[i]}, label=-100 → skip (prompt)")
        continue
    loss_i = -log_probs[i]
    total_loss += loss_i
    count += 1
    print(f"Read the values printed above and connect them to the concept in this cell.{i}: token={tokens[i]}, label={label:>3}, "
          f"log_p={log_probs[i]:.2f}, loss={loss_i:.2f}")
avg_masked = total_loss / count
print(f"Loss{avg_masked:.4f}Read the values printed above and connect them to the concept in this cell.{count}Read the values printed above and connect them to the concept in this cell.")
print()
print('Key observation: inspect the values above and connect them to the idea in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

---

### Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.ReasonRead the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Loss

In [2]:
# Teaching note: follow this line to see the main step.
print('Loss')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def reward_loss(r_chosen, r_rejected):
    diff = r_chosen - r_rejected
    prob = sigmoid(diff)
    loss = -math.log(max(prob, 1e-10))
    return loss, diff, prob

# Teaching note: follow this line to see the main step.
cases = [
    ('Read the values printed above and connect them to the concept in this cell.', 8.0, 2.0),
    ('Read the values printed above and connect them to the concept in this cell.', 6.0, 5.0),
    ('Read the values printed above and connect them to the concept in this cell.', 3.0, 7.0),
    ('Read the values printed above and connect them to the concept in this cell.', 1.0, 9.0),
]

print(f"{'Read the values printed above and connect them to the concept in this cell.':<25s} {'r_c':>6s} {'r_r':>6s} {'diff':>8s} {'σ(diff)':>10s} {'loss':>10s}")
print("-" * 70)

for desc, r_c, r_r in cases:
    loss, diff, prob = reward_loss(r_c, r_r)
    print(f"{desc:<25s} {r_c:>6.1f} {r_r:>6.1f} {diff:>8.1f} {prob:>10.4f} {loss:>10.4f}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')
print('Loss')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')

Loss
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Loss----------------------------------------------------------------------
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.LossLoss
Read the values printed above and connect them to the concept in this cell.Loss

---

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.  1. Read the values printed above and connect them to the concept in this cell.  2. Read the values printed above and connect them to the concept in this cell.  3. Read the values printed above and connect them to the concept in this cell.  4. Read the values printed above and connect them to the concept in this cell.  5. Read the values printed above and connect them to the concept in this cell.```

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

In [3]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

def ppo_loss(ratio, advantage, epsilon=0.2):
    'Loss'
    # Teaching note: follow this line to see the main step.
    unclipped = ratio * advantage
    # Teaching note: follow this line to see the main step.
    clipped = np.clip(ratio, 1-epsilon, 1+epsilon) * advantage
    # Teaching note: follow this line to see the main step.
    # Teaching note: follow this line to see the main step.
    return -min(unclipped, clipped)


ratios = [0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0]

print('Read the values printed above and connect them to the concept in this cell.')
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {"'Note'"}")
print("-" * 60)
for r in ratios:
    unclipped = r * 1.0
    clipped = min(r, 1.2) * 1.0
    loss = -min(unclipped, clipped)
    note = ""
    if r > 1.2:
        note = 'Read the values printed above and connect them to the concept in this cell.'
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {"'Note'"}")
print("-" * 60)
for r in ratios:
    unclipped = r * (-1.0)
    clipped = max(r, 0.8) * (-1.0)
    loss = -min(unclipped, clipped)
    note = ""
    if r < 0.8:
        note = 'Read the values printed above and connect them to the concept in this cell.'
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.ratio unclipped clipped loss Note------------------------------------------------------------
    0.50         0.50         0.50      -0.50 
    0.70         0.70         0.70      -0.70 
    0.90         0.90         0.90      -0.90 
    1.00         1.00         1.00      -1.00 
    1.10         1.10         1.10      -1.10 
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.ratio unclipped clipped loss Note------------------------------------------------------------
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to 

#### Loss
```
L_PPO = -E[ min(ratio × A, clip(ratio, 1-ε, 1+ε) × A) ]
        + β × KL(π_θ || π_SFT)

Among them:  ratio = π_θ(token|prompt) / π_old(token|prompt)
  A = advantage = RM_score - baseline
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

---

### Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Loss
```
L_DPO = -log( σ(
    β × log( π_θ(chosen) / π_ref(chosen) )
  - β × log( π_θ(rejected) / π_ref(rejected) )))
```

Read the values printed above and connect them to the concept in this cell.
```
LossLossRead the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

In [4]:
# Teaching note: follow this line to see the main step.
print('Loss')
print()

def dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.5):
    'Loss'
    # Teaching note: follow this line to see the main step.
    chosen_improvement = logp_chosen - logp_ref_chosen
    # Teaching note: follow this line to see the main step.
    rejected_improvement = logp_rejected - logp_ref_rejected
    
    # Teaching note: follow this line to see the main step.
    diff = beta * (chosen_improvement - rejected_improvement)
    
    # Teaching note: follow this line to see the main step.
    prob = 1 / (1 + math.exp(-diff))
    loss = -math.log(max(prob, 1e-10))
    
    return loss, chosen_improvement, rejected_improvement, diff, prob


scenarios = [
    ('Read the values printed above and connect them to the concept in this cell.',    -1.0, -5.0, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.', -1.0, -2.5, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.',     -5.0, -1.0, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.',            -0.5, -2.0, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.',            -4.5, -6.0, -3.0, -3.0),
]

print(f"{'Read the values printed above and connect them to the concept in this cell.':<28s} {'chosen_imp':>10s} {'rej_imp':>10s} {'diff':>10s} {'loss':>10s}")
print("-" * 74)

for desc, lc, lr, ref_c, ref_r in scenarios:
    loss, ci, ri, diff, prob = dpo_loss(lc, lr, ref_c, ref_r)
    print(f"{desc:<28s} {ci:>+10.2f} {ri:>+10.2f} {diff:>10.2f} {loss:>10.4f}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')

Loss
Loss--------------------------------------------------------------------------
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.LossRead the values printed above and connect them to the concept in this cell.Loss

### Read the values printed above and connect them to the concept in this cell.
|Read the values printed above and connect them to the concept in this cell.| RLHF (PPO) | DPO |
|:---|:---|:---|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
| **reward hacking** |Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Loss|
|Read the values printed above and connect them to the concept in this cell.| OpenAI (GPT-4), Anthropic (Claude) |Read the values printed above and connect them to the concept in this cell.|

Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
```
Stage 1 — Pretraining:
  LLaMA 2 Base, 2T tokens
Read the values printed above and connect them to the concept in this cell.
Stage 2 — SFT:
Read the values printed above and connect them to the concept in this cell.training 2 epochsRead the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Stage 4a — Reward Model:
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Stage 4b — PPO:
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  → LLaMA 2 Chat ✨
```

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Loss
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

---

## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.5. Loss6. Read the values printed above and connect them to the concept in this cell.7. Read the values printed above and connect them to the concept in this cell.8. Read the values printed above and connect them to the concept in this cell.9. Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.